# 5. Combining Profiles

## Purpose
Concatenate all per-well-FOV parquet files for a single patient into three
patient-level combined parquets (SC, organoid, nucleocentric).

This is **step 5 of Stage 4 (image-based profiling)**. It runs once per patient
and is typically submitted as a per-patient SLURM job.

## Inputs
- `data/{patient}/image_based_profiles/1.related_profiles/{well_fov}/`
  - `sc_profiles_{well_fov}_related.parquet`
  - `organoid_profiles_{well_fov}_related.parquet`
  - `nucleocentric_profiles_{well_fov}_related.parquet`

## Outputs
Three combined parquets in `data/{patient}/image_based_profiles/2.combined_profiles/`:

| File | Content |
|---|---|
| `sc.parquet` | All SC profiles stacked across FOVs |
| `organoid.parquet` | All organoid profiles stacked across FOVs |
| `nucleocentric.parquet` | All nucleocentric profiles stacked across FOVs |

## Notes
- Concatenation uses DuckDB `union_by_name=true`, which aligns columns by name
  rather than position. FOVs with missing columns (e.g. empty scaffold tables)
  will have those columns filled with NULL.
- Brightfield (BF) channel features are removed after concatenation as they are
  not part of the fluorescent cell painting panel and are not used in profiling.

In [1]:
import os
import pathlib

import duckdb
import pandas as pd
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
profile_base_dir = root_dir

In [2]:
if not in_notebook:
    args = parse_args()
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0014_T1"
    image_based_profiles_subparent_name = "image_based_profiles"

In [3]:
# set paths
profiles_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles"
).resolve(strict=True)
# output_paths
sc_merged_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/2.combined_profiles/sc.parquet"
).resolve()
organoid_merged_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/2.combined_profiles/organoid.parquet"
).resolve()
nucleocentric_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/2.combined_profiles/nucleocentric.parquet"
).resolve()
organoid_merged_output_path.parent.mkdir(parents=True, exist_ok=True)

In [4]:
# Discover all per-FOV parquet files under 1.related_profiles/.
# The directory structure is 1.related_profiles/{well_fov}/*.parquet,
# so one wildcard level is sufficient.
profiles = list(profiles_path.rglob("*/*.parquet"))

In [5]:
# Split files by profile type using filename prefix.
# Expected prefixes: 'sc_', 'organoid_', 'nucleocentric_'.
sc_profiles = [str(x) for x in profiles if x.name.startswith("sc_")]
organoid_profiles = [str(x) for x in profiles if x.name.startswith("organoid_")]
nucleocentric_profiles = [
    str(x) for x in profiles if x.name.startswith("nucleocentric_")
]

In [6]:
for x in nucleocentric_profiles:
    df = pd.read_parquet(x)
    if df.isnull().any().any():
        print(f"Null values found in {x}")
df

,object_id,image_set,Nucleocentric_Mito_CHAMMI75_Feature0_x,Nucleocentric_Mito_CHAMMI75_Feature1_x,Nucleocentric_Mito_CHAMMI75_Feature10_x,Nucleocentric_Mito_CHAMMI75_Feature100_x,Nucleocentric_Mito_CHAMMI75_Feature101_x,Nucleocentric_Mito_CHAMMI75_Feature102_x,Nucleocentric_Mito_CHAMMI75_Feature103_x,Nucleocentric_Mito_CHAMMI75_Feature104_x,...,Nucleocentric_ER_CHAMMI75_Feature91_y,Nucleocentric_ER_CHAMMI75_Feature92_y,Nucleocentric_ER_CHAMMI75_Feature93_y,Nucleocentric_ER_CHAMMI75_Feature94_y,Nucleocentric_ER_CHAMMI75_Feature95_y,Nucleocentric_ER_CHAMMI75_Feature96_y,Nucleocentric_ER_CHAMMI75_Feature97_y,Nucleocentric_ER_CHAMMI75_Feature98_y,Nucleocentric_ER_CHAMMI75_Feature99_y,ParentOrganoid
0,257,C10-1,2.554311,-2.499211,6.270025,1.855958,2.479886,-2.476496,-2.470967,0.746163,...,0.883319,-5.715525,-2.876581,1.559442,0.312290,0.952051,3.002643,-4.167339,-1.677402,-1
1,514,C10-1,7.105213,0.659223,1.042990,-3.127599,1.111077,1.992849,0.941766,1.696154,...,-0.892631,-6.424831,-3.211543,2.283450,-5.713564,0.695556,5.365023,-0.231492,-1.873856,1
2,771,C10-1,4.893235,0.385423,2.861571,-1.543698,2.498050,2.142777,-0.935785,-0.771587,...,0.098094,-6.672180,-2.968918,3.093965,-7.020364,1.473802,0.930933,-1.184521,-0.529366,1
3,1028,C10-1,0.144605,-4.757784,7.886053,1.660694,1.476611,-5.204203,0.042312,4.300991,...,4.270934,-7.243169,2.599591,0.495510,-0.690967,-2.965625,6.857784,2.518691,-0.112033,-1
4,1285,C10-1,0.903223,-0.710765,2.194714,1.150269,0.404250,0.147449,1.135373,-0.689170,...,3.148130,-6.224712,-2.419286,6.200053,0.547613,-0.412620,2.633893,-3.002506,1.181259,1
5,1542,C10-1,-2.210687,-1.154480,5.163727,-0.898175,-1.566604,-0.055077,-1.046951,1.293293,...,4.838858,-9.338998,3.789076,8.516677,1.679702,3.476436,4.454591,-2.240638,-0.000733,1
6,1799,C10-1,-1.201564,-2.857411,2.811934,3.594210,-2.159419,1.570191,1.565107,-0.079370,...,1.297477,-12.410695,1.356477,8.786969,-2.507699,1.640216,0.661193,-2.984402,1.019717,1
7,2056,C10-1,4.319934,2.255633,3.392022,-0.369967,1.570210,0.844378,-2.789459,-0.576255,...,-1.365574,-8.453912,-3.141807,2.626937,-4.108184,-0.731863,2.228125,-0.379013,-0.652991,1
8,2313,C10-1,2.419892,3.803269,2.523300,-4.971846,2.643979,0.927426,2.303511,1.719252,...,4.402211,-10.991883,3.374835,6.054252,-1.054146,2.006402,1.156290,-1.037703,-0.136088,1
9,2570,C10-1,-4.289413,-3.277521,3.933288,-0.819602,1.192723,4.918568,4.919585,-0.704521,...,5.504440,-5.511956,8.858730,2.981123,1.518283,0.252837,4.960571,0.586505,0.113625,1


In [7]:
# Concatenate per-FOV parquets for each profile type using DuckDB.
# union_by_name=true aligns columns by name rather than position, so FOVs with
# differing column sets (e.g. empty scaffold tables from notebook 1) are handled
# gracefully — missing columns are filled with NULL rather than causing an error.

with duckdb.connect() as conn:
    sc_profile = conn.execute(
        f"SELECT * FROM read_parquet({sc_profiles}, union_by_name=true)"
    ).df()
    organoid_profile = conn.execute(
        f"SELECT * FROM read_parquet({organoid_profiles}, union_by_name=true)"
    ).df()
    nucleocentric_profile = conn.execute(
        f"SELECT * FROM read_parquet({nucleocentric_profiles}, union_by_name=true)"
    ).df()

print(f"Single-cell profiles concatenated. Shape: {sc_profile.shape}")
print(f"Organoid profiles concatenated. Shape: {organoid_profile.shape}")
print(f"Nucleocentric profiles concatenated. Shape: {nucleocentric_profile.shape}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Single-cell profiles concatenated. Shape: (2669, 11889)
Organoid profiles concatenated. Shape: (311, 3962)
Nucleocentric profiles concatenated. Shape: (2669, 6147)


## Remove all BF channels


In [8]:
# Remove brightfield (BF) channel features from all three profile types.
# BF is a transmitted-light channel not part of the fluorescent cell painting
# panel; its features are may not meaningful for morphological profiling.
# and are interpreted differently
# Note: if no BF columns exist in the data, these drops are no-ops.

bf_cols_sc = [col for col in sc_profile.columns if "BF" in col]
sc_profile = sc_profile.drop(columns=bf_cols_sc)
print(f"SC: dropped {len(bf_cols_sc)} BF columns. Shape: {sc_profile.shape}")

bf_cols_organoid = [col for col in organoid_profile.columns if "BF" in col]
organoid_profile = organoid_profile.drop(columns=bf_cols_organoid)
print(
    f"Organoid: dropped {len(bf_cols_organoid)} BF columns. Shape: {organoid_profile.shape}"
)

bf_cols_nucleocentric = [col for col in nucleocentric_profile.columns if "BF" in col]
nucleocentric_profile = nucleocentric_profile.drop(columns=bf_cols_nucleocentric)
print(
    f"Nucleocentric: dropped {len(bf_cols_nucleocentric)} BF columns. Shape: {nucleocentric_profile.shape}"
)

SC: dropped 0 BF columns. Shape: (2669, 11889)
Organoid: dropped 0 BF columns. Shape: (311, 3962)
Nucleocentric: dropped 0 BF columns. Shape: (2669, 6147)


In [9]:
sc_profile.to_parquet(sc_merged_output_path, index=False)
organoid_profile.to_parquet(organoid_merged_output_path, index=False)
nucleocentric_profile.to_parquet(nucleocentric_profile_output_path, index=False)

In [10]:
nucleocentric_profile

,object_id,image_set,Nucleocentric_Mito_CHAMMI75_Feature0_x,Nucleocentric_Mito_CHAMMI75_Feature1_x,Nucleocentric_Mito_CHAMMI75_Feature10_x,Nucleocentric_Mito_CHAMMI75_Feature100_x,Nucleocentric_Mito_CHAMMI75_Feature101_x,Nucleocentric_Mito_CHAMMI75_Feature102_x,Nucleocentric_Mito_CHAMMI75_Feature103_x,Nucleocentric_Mito_CHAMMI75_Feature104_x,...,Nucleocentric_ER_CHAMMI75_Feature90,Nucleocentric_ER_CHAMMI75_Feature91,Nucleocentric_ER_CHAMMI75_Feature92,Nucleocentric_ER_CHAMMI75_Feature93,Nucleocentric_ER_CHAMMI75_Feature94,Nucleocentric_ER_CHAMMI75_Feature95,Nucleocentric_ER_CHAMMI75_Feature96,Nucleocentric_ER_CHAMMI75_Feature97,Nucleocentric_ER_CHAMMI75_Feature98,Nucleocentric_ER_CHAMMI75_Feature99
0,257,G8-1,0.825521,-3.740204,0.865785,0.726445,0.316764,-0.621872,-2.383686,-0.794204,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,514,G8-1,6.657349,-0.206781,1.566191,1.061814,3.645643,-1.537303,-3.186152,-1.481758,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,771,G8-1,2.616269,-4.983690,0.563332,-2.602361,0.899843,-0.182911,-0.477564,-1.679440,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1028,G8-1,7.008242,0.145953,0.891661,-2.206993,-0.072326,-0.843386,0.704497,-0.181139,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1285,G8-1,-1.694619,-1.587548,3.671193,-1.161539,-1.361440,-2.000199,3.459234,1.085015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2664,2313,C10-1,2.419892,3.803269,2.523300,-4.971846,2.643979,0.927426,2.303511,1.719252,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2665,2570,C10-1,-4.289413,-3.277521,3.933288,-0.819602,1.192723,4.918568,4.919585,-0.704521,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2666,2827,C10-1,3.386737,1.988717,-0.338405,4.546912,-0.100655,-1.621629,0.323009,0.777700,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2667,3084,C10-1,-0.894779,-1.449349,4.166935,-0.124085,-1.610468,-1.856781,1.596421,2.563449,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
